In [ ]:
# ============== CELL 1 ==============
!pip install torch torchvision tensorboard tqdm --quiet

from google.colab import files

print("Upload your KD student model (.pt file) from disk...")
uploaded = files.upload()   # pick kd_resnet18_0.8921999931335449.pt

# Take the first uploaded filename and build the path
KD_MODEL_PATH = "/content/" + list(uploaded.keys())[0]
print("KD model saved at:", KD_MODEL_PATH)

Upload your KD student model (.pt file) from disk...


Saving kd_resnet18_0.8921999931335449.pt to kd_resnet18_0.8921999931335449.pt
KD model saved at: /content/kd_resnet18_0.8921999931335449.pt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============== CELL 2 ==============
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms as tt
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
import timm
import torch.serialization

# Fix for loading timm.ResNet entire model object
torch.serialization.add_safe_globals([timm.models.resnet.ResNet])

assert os.path.exists(KD_MODEL_PATH), f"KD model not found: {KD_MODEL_PATH}"

# CIFAR-10 transforms
MEAN = [0.4914, 0.4822, 0.4465]
STD  = [0.2470, 0.2435, 0.2616]

def get_cifar10_loaders(bs=128):
    train_tf = tt.Compose([
        tt.RandomCrop(32, padding=4, padding_mode="reflect"),
        tt.RandomHorizontalFlip(),
        tt.ToTensor(),
        tt.Normalize(MEAN, STD),
    ])
    test_tf = tt.Compose([
        tt.ToTensor(),
        tt.Normalize(MEAN, STD),
    ])
    train = torchvision.datasets.CIFAR10("./data", train=True, download=True, transform=train_tf)
    test  = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)
    return (
        DataLoader(train, bs, True,  num_workers=2),
        DataLoader(test,  bs, False, num_workers=2)
    )

def count_parameters(model): return sum(p.numel() for p in model.parameters())
def model_size_mb(path): return os.path.getsize(path) / (1024*1024) if os.path.exists(path) else 0

class QuantWrapper(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.quant = torch.ao.quantization.QuantStub()
        self.m = m
        self.dequant = torch.ao.quantization.DeQuantStub()
    def forward(self, x):
        return self.dequant(self.m(self.quant(x)))

@torch.no_grad()
def eval_model(model, loader, device):
    model.eval()
    ce = nn.CrossEntropyLoss()
    total=0;correct=0;loss_sum=0
    for x,y in tqdm(loader, leave=False, desc="Eval"):
        x,y=x.to(device),y.to(device)
        out = model(x)
        loss = ce(out,y)
        loss_sum+=loss.item()
        pred = out.argmax(1)
        correct += (pred==y).sum().item()
        total += y.numel()
    return correct/total, loss_sum/len(loader)

class QATTrainer:
    def __init__(self, epochs=100):
        self.epochs = epochs
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print("Using device:", self.device)

        self.train_ld, self.test_ld = get_cifar10_loaders()

        # -------- FIX: safe unpickling --------
        print("Loading KD student:", KD_MODEL_PATH)
        float_model = torch.load(
            KD_MODEL_PATH,
            map_location="cpu",
            weights_only=False      # IMPORTANT FIX
        )
        float_model.train()

        print("\n==== FLOAT MODEL (before QAT) ====")
        print(float_model)
        print("Parameters:", count_parameters(float_model))

        qat_model = QuantWrapper(float_model)
        qat_model.qconfig = torch.ao.quantization.get_default_qat_qconfig("fbgemm")
        torch.ao.quantization.prepare_qat(qat_model, inplace=True)

        self.model_float = float_model.to(self.device)
        self.model_qat   = qat_model.to(self.device)

        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.SGD(self.model_qat.parameters(), lr=1e-3, momentum=0.9, weight_decay=1e-4)
        self.scheduler = optim.lr_scheduler.MultiStepLR(self.optimizer, milestones=[50,75], gamma=0.1)
        self.writer = SummaryWriter("runs/qat")

    def train_epoch(self, epoch):
        self.model_qat.train()
        running=0
        loop=tqdm(self.train_ld, desc=f"[QAT] Epoch {epoch+1}/{self.epochs}")
        for i,(x,y) in enumerate(loop):
            x,y=x.to(self.device),y.to(self.device)
            self.optimizer.zero_grad()
            out=self.model_qat(x)
            loss=self.criterion(out,y)
            loss.backward()
            self.optimizer.step()
            running+=loss.item()
            loop.set_postfix(loss=running/(i+1))
        return running/len(self.train_ld)

    def run(self):
        best=0
        float_path="/content/qat_float.pth"
        quant_path="/content/qat_int8.pth"

        torch.save(self.model_float.state_dict(), float_path)

        for e in range(self.epochs):
            print(f"\n====== Epoch {e+1}/{self.epochs} ======")
            train_loss=self.train_epoch(e)
            self.scheduler.step()
            acc,val_loss=eval_model(self.model_qat,self.test_ld,self.device)
            print(f"Val Acc: {acc:.4f}")

            if acc>best:
                best=acc
                print(">>> BEST MODEL UPDATED (INT8)")
                q= torch.ao.quantization.convert(self.model_qat.cpu().eval(), inplace=False)
                torch.save(q.state_dict(), quant_path)

                print("\n===== QUANTIZED MODEL STATS =====")
                print(q)
                print("Params:", count_parameters(q))
                print("Float size:", model_size_mb(float_path),"MB")
                print("INT8 size:", model_size_mb(quant_path),"MB")

                self.model_qat.to(self.device)

trainer=QATTrainer(epochs=100)
trainer.run()

Using device: cuda


100%|██████████| 170M/170M [00:03<00:00, 55.3MB/s]


Loading KD student: /content/kd_resnet18_0.8921999931335449.pt

==== FLOAT MODEL (before QAT) ====
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (act1): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (drop_block): Identity()
      (act1): ReLU(inplace=True)
      (aa): Identity()
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act2): ReLU(inplace=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kerne

/tmp/ipython-input-135540459.py:91: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  torch.ao.quantization.prepare_qat(qat_model, inplace=True)
/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/observer.py:246: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future relea


====== Epoch 1/100 ======


...

Val Acc: 0.8904

====== Epoch 99/100 ======


[QAT] Epoch 99/100: 100%|██████████| 391/391 [00:36<00:00, 10.69it/s, loss=0.138]


Val Acc: 0.8916

====== Epoch 100/100 ======


[QAT] Epoch 100/100: 100%|██████████| 391/391 [00:36<00:00, 10.73it/s, loss=0.137]
                                                     

Val Acc: 0.8916
